# Data Quality Findings

This notebook profiles the Silver layer and documents the main findings used to justify Gold governance KPIs.

In [1]:
from pathlib import Path
import ast
import json

import pandas as pd

silver_dir = Path('../datalake_silver')
files = sorted(silver_dir.glob('*.parquet'))
frames = [pd.read_parquet(path) for path in files]
silver = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
silver.head()

,newsLink,comments,_id,snapshot_id,snapshot_date,type,id,url,twitterUrl,text,...,quoted_tweet_quoted_tweet_card,quoted_tweet_quoted_tweet_quoted_tweet,quoted_tweet_quoted_tweet_retweeted_tweet,quoted_tweet_quoted_tweet_isLimitedReply,quoted_tweet_quoted_tweet_communityInfo,quoted_tweet_quoted_tweet_article,author_profile_bio_entities_description_symbols,entities_timestamps,quoted_tweet_author_profile_bio_entities_description_hashtags,quoted_tweet_author_profile_bio_entities_url_urls
0,https://www.gsmarena.com/xiaomi_unveils_smart_...,"[""I like how at least some still make earbuds ...",6a19f676d5d72db4fa2cf569,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1,https://www.gsmarena.com/samsung_is_now_shippi...,"[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...",6a19f676d5d72db4fa2cf56a,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
2,https://www.gsmarena.com/samsung_galaxy_z_fold...,"[""Waiting for the day Samsung increases batter...",6a19f676d5d72db4fa2cf56b,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3,https://www.gsmarena.com/vertu_unveils_alphafo...,"[""Rebranded zte with leather slapped onto it. ...",6a19f676d5d72db4fa2cf56c,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
4,https://www.gsmarena.com/hmd_grand_renders_and...,"[""-, 28 May 20261. How did you get banned in u...",6a19f676d5d72db4fa2cf56d,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN


In [2]:
if not silver.empty:
    null_rates = silver.isna().mean().sort_values(ascending=False).to_frame('null_rate')
    display(null_rates.head(15))

    numeric_cols = silver.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        describe = silver[numeric_cols].describe().T
        display(describe)

    if 'text' in silver.columns:
        text_lengths = silver['text'].astype(str).str.len()
        display(text_lengths.describe())
else:
    print('No Silver data found.')

,null_rate
inReplyToId,1.0
author_verifiedType,1.0
quoted_tweet_article,1.0
quoted_tweet_card,1.0
quoted_tweet_author_verifiedType,1.0
quoted_tweet_inReplyToUsername,1.0
quoted_tweet_inReplyToUserId,1.0
quoted_tweet_inReplyToId,1.0
article,1.0
author_automatedBy,1.0


,count,mean,std,min,25%,50%,75%,max
retweetCount,120.0,2.716667,9.128571e+00,0.0,0.00,0.0,0.00,53.0
replyCount,120.0,5.083333,1.873324e+01,0.0,0.00,0.0,1.00,132.0
likeCount,120.0,94.266667,3.668576e+02,0.0,0.00,0.0,2.25,2013.0
quoteCount,120.0,0.283333,1.456272e+00,0.0,0.00,0.0,0.00,11.0
viewCount,120.0,6704.900000,2.730561e+04,2.0,21.00,43.0,260.75,152549.0
bookmarkCount,120.0,6.650000,2.650183e+01,0.0,0.00,0.0,0.00,156.0
author_followers,120.0,9751.233333,3.979350e+04,11.0,111.75,477.0,2874.25,295464.0
author_following,120.0,1120.866667,1.797587e+03,0.0,194.50,431.0,1028.50,7405.0
author_fastFollowersCount,120.0,0.000000,0.000000e+00,0.0,0.00,0.0,0.00,0.0
author_favouritesCount,120.0,47608.433333,7.562567e+04,0.0,1631.25,13849.5,54113.50,423507.0


count     120.000000
mean      265.866667
std       195.731291
min        70.000000
25%       145.750000
50%       246.500000
75%       285.250000
max      1070.000000
Name: text, dtype: float64

In [3]:
findings = []
if not silver.empty:
    if 'lang' in silver.columns:
        findings.append(('language_distribution', silver['lang'].value_counts(dropna=False).head(10)))
    if 'comments' in silver.columns:
        comments = silver['comments'].astype(str).map(lambda x: len(ast.literal_eval(x)) if x.startswith('[') and x.endswith(']') else 0)
        findings.append(('comment_count', comments.describe()))

    for title, value in findings:
        print(f'## {title}')
        display(value)
else:
    print('No findings generated.')

AttributeError: 'float' object has no attribute 'startswith'